In [1]:
import numpy as np
import sys
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
sys.path.append('../../')
from src.models.vectorDBbuild import VectorDBBuilder
from src.models.lightfm import LightFMTrainer
from src.models.hybrid_retrieval import RetrievalEngine
from src.models.newUser import NewUserRetriever

def run_full_pipeline(data, test_user_ids=None, debug=False):
    print("Training LightFM model")
    lightfm_model = LightFMTrainer()
    lightfm_model.fit_and_train(data, learning_rate=0.01, epochs=50)
    all_recommendations = lightfm_model.get_top_k(k=50)
    
    df_data = []
    for user_id, recs in all_recommendations.items():
        df_data.append({
            'user_id': user_id,
            'recommendations': recs
        })
    
    df = pd.DataFrame(df_data)
    
    # for user_id, recs in all_recommendations.items():
    #     for rec in recs:
    #         # Enhance with additional metadata if available
    #         enhanced_rec = {
    #             'user_id': user_id,
    #             'iid': rec['iid'],
    #             'title': rec.get('title', ''),
    #             'categories': rec.get('categories', []),
    #             'description': rec.get('description', ''),
    #             'cf_score': rec['score']
    #         }
    #         df_data.append(enhanced_rec)
    
    # df = pd.DataFrame(df_data)
    
    # build vector DB
    print("Building vector database...")
    vector_builder = VectorDBBuilder()
    embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    df = vector_builder.preprocess_for_augmentation(df)
    items_df = vector_builder.extract_items_metadata(df)
    chroma_db = vector_builder.build_from_dataframe(items_df)
    
    # set up retrieval engine 
    retriever = RetrievalEngine(None, embedder)
    retriever.chroma_db = chroma_db
    # llm_reranker = LLMReranker()
    
    return {
        'df': df,
        'chroma_db': chroma_db,
        'retriever': retriever,
        # 'llm_reranker': llm_reranker,
        # 'top_k_results': top_k_results,
        'lightfm_model': lightfm_model
    }

/Users/vynguyen/anaconda3/envs/thesis/lib/python3.11/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [2]:
def get_recommendations_for_user(pipeline_objects, user_id, query, debug=False):
    df = pipeline_objects['df']
    chroma_db = pipeline_objects['chroma_db']
    retriever = pipeline_objects['retriever']
    # llm_reranker = pipeline_objects['llm_reranker']
    #top_k_results = pipeline_objects['top_k_results']
    
    user_type = "existing" if check_user_type(user_id, df) else "new"
    
    if user_type == "existing":
        user_recs = df[df['user_id'] == user_id]['recommendations'].iloc[0]
        # user_recs = top_k_results[user_id]
        
        augmented = []
        for rec in user_recs:
            # get additional context from vector database
            try:
                chunks = chroma_db.similarity_search(
                    query=rec.get('combined_text', rec.get('title', '')),
                    k=3,
                    filter={"iid": rec['iid']}
                )
                augmented_text = f"{rec.get('combined_text', '')}\nRelated Context:\n{' '.join([chunk.page_content for chunk in chunks])}"
            except:
                # fallback if vector search fails
                augmented_text = rec.get('combined_text', '')
            
            augmented.append({
                **rec,
                "combined_text": rec.get('combined_text', ''),
                "augmented_text": augmented_text
            })
        
        # if debug:
        #     print("=== After Augmentation ===")
        #     print(f"Found {len(augmented)} items")
        #     for i, item in enumerate(augmented[:3], 1):
        #         print(f"{i}. {item['title']} - {item.get('combined_text', '')[:100]}...")
        
        # Hybrid retrieval
        scored = retriever.hybrid_retrieve(augmented, query)
        
        if debug:
            print("\n=== After Hybrid Retrieval ===")
            print(f"Top 10 items by hybrid score:")
            for i, item in enumerate(sorted(scored, key=lambda x: x.get('hybrid_score', 0), reverse=True)[:10], 1):
                print(f"{i}. {item['title']} (Hybrid: {item.get('hybrid_score', 0):.3f})")
                print(f"   Categories: {''.join(item.get('categories', []))}")
                description = item.get('description', '')
                if description and isinstance(description, str):
                    print(f"   Description: {description[:150]}...")
                # print(f"   Description: {item.get('description', '')[:150]}...\n")
        
        # MMR Diversification
        diversified = retriever.mmr_diversify(scored, query)
        
        if debug:
            print("\n=== After MMR Diversification ===")
            print(f"Top 10 diversified items:")
            for i, item in enumerate(diversified[:10], 1):
                print(f"{i}. {item['title']} (Hybrid: {item.get('hybrid_score', 0):.3f}, MMR: {item.get('mmr_score', 0):.3f})")
                print(f"   Categories: {''.join(item.get('categories', []))}")
                # print(f"   Description: {item.get('description', '')[:150]}...\n")
                description = item.get('description', '')
                if description and isinstance(description, str):
                    print(f"   Description: {description[:150]}...")
                print()
                
        # LLM Reranking
        # llm_result = llm_reranker.rerank(diversified, query, user_id, user_type)
        
        # if debug:
        #     print("\n=== Final LLM-Ranked Results ===")
        #     print(f"Top 10 final recommendations:")
        #     for i, item in enumerate(llm_result['parsed_items'][:10], 1):
        #         print(f"{i}. {item['title']} (ID: {item['iid']})")
        #         if 'hybrid_score' in item:
        #             print(f"   Hybrid Score: {item['hybrid_score']:.3f}")
        #         if 'mmr_score' in item:
        #             print(f"   MMR Score: {item['mmr_score']:.3f}")
        #         print("   Reason:")
        #         if 'llm_reason' in item:
        #             for reason_line in item['llm_reason'].split('\n'):
        #                 print(f"     {reason_line}")
        #         else:
        #             print("     - No reasoning provided")
        #         print() 
        
        return diversified[:10]
        # llm_result['parsed_items'][:10]

In [3]:
def check_user_type(user_id, df):
    return user_id in df['user_id'].values

In [4]:
import pickle
import os

def save_trained_pipeline_simple(pipeline_objects, save_dir="saved_pipeline"):
    """Save only the essential data needed for evaluation"""
    os.makedirs(save_dir, exist_ok=True)
    
    # Extract and save only the recommendations DataFrame
    df = pipeline_objects['df']
    
    # Save the DataFrame to CSV (more reliable than pickle)
    df.to_csv(f"{save_dir}/recommendations.csv", index=False)
    
    # Save any additional metadata needed
    metadata = {
        'total_users': len(df),
        'columns': list(df.columns)
    }
    
    with open(f"{save_dir}/metadata.pkl", 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f" Pipeline data saved to {save_dir}/")
    print(f" - Recommendations: {len(df)} users")
    print(f" - File: {save_dir}/recommendations.csv")

def load_trained_pipeline_simple(save_dir="saved_pipeline", chroma_db_path="./chroma_db"):
    """Load pipeline data for evaluation"""
    try:
        # Load recommendations
        df = pd.read_csv(f"{save_dir}/recommendations.csv")
        
        # Convert recommendations column from string back to list of dicts
        if 'recommendations' in df.columns and isinstance(df['recommendations'].iloc[0], str):
            import ast
            df['recommendations'] = df['recommendations'].apply(ast.literal_eval)
        
        # Load ChromaDB
        from langchain.vectorstores import Chroma
        from langchain.embeddings import HuggingFaceEmbeddings
        
        embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
        chroma_db = Chroma(persist_directory=chroma_db_path, embedding_function=embedder)
        
        # Recreate retriever
        from src.models.hybrid_retrieval import RetrievalEngine
        retriever = RetrievalEngine(None, embedder)
        retriever.chroma_db = chroma_db
        
        pipeline_objects = {
            'df': df,
            'chroma_db': chroma_db,
            'retriever': retriever
        }
        
        print(f" Pipeline loaded from {save_dir}/")
        print(f"   - Loaded recommendations for {len(df)} users")
        return pipeline_objects
        
    except Exception as e:
        print(f" Error loading pipeline: {e}")
        return None

def is_pipeline_saved(save_dir="saved_pipeline"):
    """Check if pipeline is already saved"""
    return os.path.exists(f"{save_dir}/pipeline_objects.pkl")

In [7]:
# train_pipeline.py
import pandas as pd
from src.models.lightfm import LightFMTrainer
from src.models.vectorDBbuild import VectorDBBuilder
from src.models.hybrid_retrieval import RetrievalEngine
from langchain.embeddings import HuggingFaceEmbeddings

def train_and_save_pipeline():
    """Train the pipeline once and save it"""
    print(" TRAINING PIPELINE (One-time process)")
    print("=" * 50)
    
    # Load only train data
    train_data = pd.read_csv("train_set.csv")
    print(f"Training data: {train_data.shape[0]} rows, {train_data['uid'].nunique()} users")
    
    # Train the model
    print("Training LightFM model...")
    lightfm_model = LightFMTrainer()
    lightfm_model.fit_and_train(train_data, learning_rate=0.01, epochs=50)
    
    # Generate recommendations for ALL train users
    print("Generating recommendations...")
    all_recommendations = lightfm_model.get_top_k(k=50)
    
    # Build the DataFrame structure
    df_data = []
    for user_id, recs in all_recommendations.items():
        df_data.append({
            'user_id': user_id,
            'recommendations': recs
        })
    
    df = pd.DataFrame(df_data)
    
    # Build vector database
    print("Building vector database...")
    vector_builder = VectorDBBuilder()
    embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    df = vector_builder.preprocess_for_augmentation(df)
    items_df = vector_builder.extract_items_metadata(df)
    chroma_db = vector_builder.build_from_dataframe(items_df)
    
    # Set up retrieval engine
    retriever = RetrievalEngine(None, embedder)
    retriever.chroma_db = chroma_db
    
    # Create pipeline objects
    pipeline_objects = {
        'df': df,
        'chroma_db': chroma_db,
        'retriever': retriever,
        # 'lightfm_model': lightfm_model  # Keep in memory for this session only
    }
    
    # Save the trained pipeline (using simple approach)
    save_trained_pipeline_simple(pipeline_objects)
    
    print(" Training completed and pipeline saved!")
    return pipeline_objects

if __name__ == "__main__":
    train_and_save_pipeline()

 TRAINING PIPELINE (One-time process)
Training data: 98597 rows, 57297 users
Training LightFM model...
Generating recommendations...


100%|███████████████████████████████████████████████████████████████████████████████████████| 57297/57297 [9:03:59<00:00,  1.76it/s]
/Users/vynguyen/Documents/VSCode/Thesis/notebooks/amazon/../../src/models/vectorDBbuild.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)


Building vector database...
 Pipeline data saved to saved_pipeline/
 - Recommendations: 57297 users
 - File: saved_pipeline/recommendations.csv
 Training completed and pipeline saved!


In [6]:
from sklearn.metrics import ndcg_score
import numpy as np

def calculate_relevance_metrics(true_items, recommendations, k=10, debug=False):
    predicted_items = [rec['iid'] for rec in recommendations[:k]]
    true_set = set(true_items)

    relevance_score = [1 if iid in true_set else 0 for iid in predicted_items]
    
    # hit rate
    hit = 1 if any(relevance_score) else 0

    # MRR@k
    mrr = 0.0
    for rank, rel in enumerate(relevance_scores, start=1):
        if rel == 1:
            mrr = 1.0 / rank
            break
            
    # Precision@k
    precision = sum(relevance_score)/k

    # AP@k
    ap = 0.0
    num_relevant = 0
    for rank, rel in enumerate(relevance_score, start = 1):
        if rel == 1:
            num_relevant += 1
            ap += num_relevant/rank
    ap = ap/min(len(true_set), k) if true_set else 0

    # NDCG@k (using binary relevance)
    ideal_relevance = sorted(relevance_score, reverse=True)
    if sum(relevance_true) == 0:
        ndcg = 0.0
    else:
        ndcg = ndcg_score([ideal_relevance], [relevance_score], k=k)
    
    if debug:
        print(f"True items: {true_items}")
        print(f"Predicted items: {predicted_items}")
        print(f"Relevance scores: {relevance_score}")
        print(f"Hit: {hit}, Precision@{k}: {precision:.4f}, Recall@{k}: {recall:.4f}")
        print(f"AP@{k}: {ap:.4f}, NDCG@{k}: {ndcg:.4f}, MRR@{k}: {mrr:.4f}")
        
    return {
        f'HitRate@{k}': hit,
        f'AP@{k}': ap,
        f'NDCG@{k}': ndcg,
        f'MRR@{k}': mrr,
        'num_relevant': sum(relevance_score),
        'num_true_items': len(true_set)
    }

In [7]:
def calculate_aggregate_relevance_metrics(user_metrics_list, k=10):
    """
    Calculate aggregate metrics across all users
    """
    if not user_metrics_list:
        return {}
    
    # Convert to DataFrame for easier aggregation
    import pandas as pd
    df = pd.DataFrame(user_metrics_list)
    
    aggregate_metrics = {
        f'MAP@{k}': df[f'AP@{k}'].mean(),
        f'Mean_NDCG@{k}': df[f'NDCG@{k}'].mean(),
        f'Mean_MRR@{k}': df[f'MRR@{k}'].mean(),
        f'HitRate@{k}': df[f'HitRate@{k}'].mean(),
        'Total_Users': len(df),
        'Total_Relevant_Items': df['num_relevant'].sum(),
        'Avg_True_Items_Per_User': df['num_true_items'].mean()
    }
    
    # # Add standard deviations
    # aggregate_metrics.update({
    #     f'Std_NDCG@{k}': df[f'NDCG@{k}'].std(),
    #     f'Std_Precision@{k}': df[f'Precision@{k}'].std(),
    #     f'Std_Recall@{k}': df[f'Recall@{k}'].std()
    # })
    
    return aggregate_metrics

In [8]:
def calculate_aggregate_diversity_metrics(all_diversity_metrics):
    """
    Calculate aggregate diversity metrics across all users
    """
    if not all_diversity_metrics:
        return {}
    
    df = pd.DataFrame(all_diversity_metrics)
    
    return {
        'Mean_CategoryCoverage': df['CategoryCoverage'].mean(),
        'Mean_UniqueCategories': df['UniqueCategories'].mean(),
        'Mean_Entropy': df['Entropy'].mean(),
        'Mean_GiniIndex': df['GiniIndex'].mean(),
        'Std_CategoryCoverage': df['CategoryCoverage'].std(),
        'Std_UniqueCategories': df['UniqueCategories'].std()
    }

In [ ]:
# evaluate_pipeline.py
import pandas as pd
# from src.evaluation.relevance_metrics import calculate_user_relevance_metrics
from src.evaluation.diversity_metrics import calculate_diversity_metrics, clean_categories
# from src.evaluation.aggregate_metrics import (
#     calculate_aggregate_relevance_metrics, 
#     calculate_aggregate_diversity_metrics
# )

def evaluate_with_saved_pipeline(test_query="pop music with piano", sample_size=None):
    """
    Evaluate using saved pipeline - NO RETRAINING NEEDED!
    """
    print("EVALUATION WITH SAVED PIPELINE")
    print("=" * 50)
    
    # Load data
    train_data = pd.read_csv("train_set.csv")
    test_data = pd.read_csv("test_set.csv")
    
    print(f"Test data: {test_data.shape[0]} rows, {test_data['uid'].nunique()} users")
    
    # Load saved pipeline
    pipeline_objects = load_trained_pipeline()
    if pipeline_objects is None:
        print("No saved pipeline found. Please run train_pipeline.py first.")
        return
    
    # Get qualified test users (users that exist in both train and test)
    train_users = set(train_data['uid'].unique())
    test_users = test_data['uid'].unique().tolist()
    qualified_users = [user for user in test_users if user in train_users]
    
    if sample_size and sample_size < len(qualified_users):
        qualified_users = qualified_users[:sample_size]
    
    print(f"Evaluating {len(qualified_users)} qualified test users...")
    
    # Store metrics
    all_relevance_metrics = []
    all_diversity_metrics = []
    
    for i, user_id in enumerate(qualified_users):
        print(f"  User {i+1}/{len(qualified_users)}: {user_id}", end=" ")
        
        try:
            # Get recommendations from saved pipeline
            recommendations = get_recommendations_for_user(
                pipeline_objects, user_id, test_query, debug=False
            )
            
            if not recommendations:
                print("No recommendations")
                continue
            
            # Get ground truth from test data
            true_items = test_data[test_data['uid'] == user_id]['iid'].unique().tolist()
            
            if not true_items:
                print("No ground truth")
                continue
            
            # Calculate metrics
            relevance_metrics = calculate_user_relevance_metrics(true_items, recommendations)
            diversity_metrics = calculate_diversity_metrics(recommendations)
            
            # Store results
            relevance_metrics['user_id'] = user_id
            diversity_metrics['user_id'] = user_id
            
            all_relevance_metrics.append(relevance_metrics)
            all_diversity_metrics.append(diversity_metrics)
            
            print(f"→ (P@10: {relevance_metrics['Precision@10']:.3f}, NDCG@10: {relevance_metrics['NDCG@10']:.3f})")
            
        except Exception as e:
            print(f"→ Error: {str(e)[:50]}... ")
    
    # Calculate aggregate results
    print("\n" + "=" * 50)
    print("FINAL RESULTS")
    print("=" * 50)
    
    if all_relevance_metrics:
        aggregate_relevance = calculate_aggregate_relevance_metrics(all_relevance_metrics)
        aggregate_diversity = calculate_aggregate_diversity_metrics(all_diversity_metrics)
        
        print(f"\n RELEVANCE METRICS ({len(all_relevance_metrics)} users):")
        print(f"MAP@10: {aggregate_relevance['MAP@10']:.4f}")
        print(f"Mean NDCG@10: {aggregate_relevance['Mean_NDCG@10']:.4f}")
        print(f"Mean Precision@10: {aggregate_relevance['Mean_Precision@10']:.4f}")
        print(f"Hit Rate@10: {aggregate_relevance['HitRate@10']:.4f}")
        
        print(f"\n DIVERSITY METRICS:")
        print(f"Mean Category Coverage: {aggregate_diversity['Mean_CategoryCoverage']:.4f}")
        print(f"Mean Unique Categories: {aggregate_diversity['Mean_UniqueCategories']:.2f}")
        
        # Save results
        save_evaluation_results(all_relevance_metrics, all_diversity_metrics)
        
    else:
        print(" No successful evaluations!")

def quick_evaluation(test_query="pop music", user_sample=5):
    """
    Quick evaluation for testing
    """
    print("QUICK EVALUATION")
    return evaluate_with_saved_pipeline(test_query=test_query, sample_size=user_sample)

def save_evaluation_results(relevance_metrics, diversity_metrics):
    """Save evaluation results"""
    import pandas as pd
    
    rel_df = pd.DataFrame(relevance_metrics)
    div_df = pd.DataFrame(diversity_metrics).add_prefix('div_')
    
    results_df = rel_df.merge(div_df, left_on='user_id', right_on='div_user_id')
    results_df = results_df.drop(['div_user_id'], axis=1)
    
    results_df.to_csv("evaluation_results.csv", index=False)
    print(f"Results saved to: evaluation_results.csv")

if __name__ == "__main__":
    # Choose evaluation mode:
    
    # Option 1: Quick test (fast)
    quick_evaluation(user_sample=5)
    
    # Option 2: Full evaluation
    # evaluate_with_saved_pipeline(sample_size=None)

QUICK EVALUATION
EVALUATION WITH SAVED PIPELINE
Test data: 172 rows, 153 users
